# Phase 6: Abstractive Summarization - Knowledge Distillation into CapsuleRoute

## Task
Abstractive summarization of long legal judgments. Scored with BERTScore F1 against the reference summaries.

Model: a sentence selector picks the important sentences; a CapsuleRoute student is fully fine-tuned to rewrite them, using the gold pairs plus teacher-written pairs from p6_teacher_pairs.json.

## Input
- Phase-6 data (train documents with summaries, test documents); downloaded from Hugging Face if not attached.
- CapsuleRoute checkpoint and tokenizer (found in /kaggle/input).
- Teacher pairs file p6_teacher_pairs.json.

## Flow of the notebook (from input to output)
1. Write the main pipeline p6_core.py: data loading, sentence splitting, sentence features and selector, fine-tuning methods, training, generation, summary assembly and scoring.
2. Write the CapsuleRoute model code.
3. Load the model and run the pipeline: split documents into sentences, train a sentence selector on simple features, build (sentence, summary-sentence) training pairs, fine-tune the model with full fine-tuning and the teacher pairs to rewrite chosen sentences, try generation and assembly settings on dev documents, and write the test summaries with the best settings.

## Output
- Submission zips with the test summaries (dev score in the file name).
- One zip with all outputs.

In [1]:
%%writefile p6_core.py
# Main summarization code: data, model, training (saved as p6_core.py)
import os, re, json, glob, math, time, random, zipfile, subprocess, sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter

REPO = "Exploration-Lab/CS-781-Capstone"
DEV = "cuda" if torch.cuda.is_available() else "cpu"
AB = r"\b(?:No|Nos|v|vs|Sec|Art|Dr|Mr|Mrs|Ms|Smt|Shri|Ltd|Pvt|Co|St|Hon|Rs|Cr|Crl|Ors|Anr|Addl|Dy|Ch|Vol|Para|Cl|Ex|Rev|viz|cf)\."
DEFAULT = dict(src=112, gen=80, align_min=0.25, sel_min=0.15, max_pairs=150000, dev_docs=120, lr=3e-4, lr_cond=1e-3, wd=0.01, warm=100, eval=500, bs=32, gbs=128, minlen=6, nbl=3,
               maxw=400, slack=1.4, seed=0, smoke=0, out="/kaggle/working", r=16, alpha=32, adapter_r=64, epochs=3, hours=5.0, teacher="", dump_teacher=0, resume=False, fast=False,
               mmrs=(0.3, 0.5, 0.7), decodes=((0.0, 1.2), (2.0, 1.2), (4.0, 1.2), (3.0, 1.0)), ths=(0.0, 0.2, 0.3, 0.4, 0.5), stricts=(0, 1), keep_test=2, cons_c=(0.5, 2.0), cons_ths=(0.0, 0.2, 0.4))

# It takes path (file path) and pats (file patterns) as input, finds the right input file to use and returns a path of the chosen file.
def pick(path, pats):
    if path and os.path.exists(path):
        print("using", path); return path
    for p in pats:
        h = sorted(x for x in glob.glob("/kaggle/input/**/" + p, recursive=True) if os.path.isfile(x) and "trainable" not in x and "resume" not in x)
        if h:
            print("auto-selected", h[0], "| other matches:", h[1:4]); return h[0]
    raise FileNotFoundError(f"set the path variable; files seen: {[x for x in glob.glob('/kaggle/input/**/*', recursive=True) if os.path.isfile(x)][:60]}")

# It takes name and keys as input, searches the input folder for a data file and returns a file path (or None if not found).
def find(name, keys=("phase_6", "phase6", "summ")):
    h = [x for r in ("/kaggle/input", "/kaggle/working", ".") for x in glob.glob(f"{r}/**/{name}", recursive=True) if any(k in x.lower() for k in keys)]
    return min(h, key=len) if h else None

# It takes path (file path) as input, loads a JSON file and returns a list of records.
def load(path):
    raw = json.load(open(path, encoding="utf-8"))
    if isinstance(raw, dict):
        raw = [{**v, "id": v.get("id", k)} if isinstance(v, dict) else {"id": k, "document": v} for k, v in raw.items()]
    g = lambda r, ks: next((r[k] for k in ks if r.get(k) is not None), "")
    fix = lambda t: re.sub(r"\s+", " ", " ".join(map(str, t)) if isinstance(t, list) else str(t)).strip()
    print("fields of the first record:", list(raw[0].keys()), flush=True)
    return [dict(id=r.get("id"), doc=fix(g(r, ("document", "text", "judgment", "case_text"))), sum=fix(g(r, ("summary", "reference", "headnote", "abstract")))) for r in raw]

# It takes dest (download folder) as input, loads the data, downloading it if needed and returns (train records, test records).
def get_data(dest="/kaggle/working/phase6_data"):
    p = {n: find(n) for n in ("train.json", "test.json")}
    if not all(p.values()):
        from huggingface_hub import HfApi, hf_hub_download, login
        try:
            from kaggle_secrets import UserSecretsClient
            login(token=UserSecretsClient().get_secret("HF_TOKEN"))
        except Exception:
            login(token=os.environ.get("HF_TOKEN"))
        fs = HfApi().list_repo_files(REPO, repo_type="dataset")
        for n in p:
            c = [f for f in fs if f.lower().endswith("/" + n) and ("phase_6" in f.lower() or "summ" in f.lower())]
            assert c, f"{n} not found; files: {fs[:20]}"
            p[n] = hf_hub_download(REPO, min(c, key=len), repo_type="dataset", local_dir=dest)
    return load(p["train.json"]), load(p["test.json"])

# It takes t (text) as input, splits text into sentences and returns a list of sentences.
def sents(t):
    t = re.sub(AB, lambda m: m.group()[:-1] + "\x00", t, flags=re.I)
    t = re.sub(r"\b([A-Z])\.", "\\1\x00", t)
    t = re.sub(r"(\d)\.(?=\d)", "\\1\x00", t)
    return [p.replace("\x00", ".").strip() for p in re.split(r"(?<=[.!?])\s+(?=[\"'(\[]?[A-Z0-9])", t) if len(p.split()) >= 3]

CUE = re.compile(r"\b(held|hold|court|appeal|appellant|respondent|dismiss\w*|allowed|contend\w*|submit\w*|petition\w*|judgment|order|section|act|rule|article|decree|tribunal|assessee|conclu\w+|therefore|accordingly)\b", re.I)

# It takes S (sentences) and X (TF-IDF matrix of the sentences) as input, returns a feature matrix; it is the simple features for each sentence.
def feats(S, X):
    n = len(S); c = np.asarray(X.mean(0)).ravel(); c = c / (np.linalg.norm(c) + 1e-9); i = np.arange(n)
    return np.c_[X @ c, i / n, np.minimum(i, 40), np.minimum(n - 1 - i, 40), [math.log1p(len(s.split())) for s in S], [sum(ch.isdigit() for ch in s) / len(s) for s in S], np.asarray(X.sum(1)).ravel()]

# It takes S (sentences) and X (TF-IDF matrix of the sentences) as input, returns a feature matrix; it is the more features for each sentence.
def feats2(S, X):
    n = len(S); G = (X @ X.T).toarray(); np.fill_diagonal(G, 0); c = np.asarray(X.mean(0)).ravel(); c = c / (np.linalg.norm(c) + 1e-9); cs = X @ c
    f3 = np.asarray(X[:3].mean(0)).ravel(); f3 = f3 / (np.linalg.norm(f3) + 1e-9); wc = np.array([len(s.split()) for s in S], float)
    return np.c_[feats(S, X), G.mean(1), G.max(1) if n > 1 else np.zeros(n), np.argsort(np.argsort(cs)) / max(n - 1, 1), wc, wc / (wc.mean() + 1e-9), [sum(ch.isupper() for ch in s) / len(s) for s in S],
                 [len(CUE.findall(s)) for s in S], [s.count('"') + s.count("'") for s in S], np.full(n, math.log(n)), X @ f3]

class Sel:
    # It takes kind (type) and model as input, stores the sentence selection model and returns nothing.
    def __init__(self, kind, model):
        self.kind, self.m = kind, model

    # It takes f (sentence features) as input, scores how important each sentence is and returns an importance score per sentence.
    def score(self, f):
        return self.m.predict(f)[:] if self.kind == "reg" else self.m.predict_proba(f[:, :7] if self.kind == "v1" else f)[:, 1]

# It takes tr (training data), te (test data) and cfg (settings) as input, splits sentences, builds features and trains the sentence selector and returns the prepared data, selectors and settings.
def prepare(tr, te, cfg):
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.ensemble import HistGradientBoostingClassifier as HC, HistGradientBoostingRegressor as HR
    random.Random(0).shuffle(tr)
    nd = min(cfg["dev_docs"], max(8, len(tr) // 30)); dv, tr = tr[:nd], tr[nd:]
    for d in tr + dv + te:
        d["S"], d["R"] = sents(d["doc"]), sents(d["sum"]) if d["sum"] else []
    txt = [s for d in random.Random(1).sample(tr, min(len(tr), 3000)) for s in d["S"] + d["R"]]
    tv = TfidfVectorizer(sublinear_tf=True, stop_words="english", min_df=2, max_features=80000).fit(txt)
    Fm, Lb, Ys, nm = [], [], [], []
    for k, d in enumerate(tr + dv):
        d["P"], d["P0"] = [], []
        if not d["S"] or not d["R"]:
            continue
        X = tv.transform(d["S"]); sim = (tv.transform(d["R"]) @ X.T).toarray(); b = sim.argmax(1)
        d["P"] = [(j, int(b[j])) for j in range(len(b)) if sim[j, b[j]] >= cfg["align_min"]]
        d["P0"] = [(j, int(b[j])) for j in range(len(b)) if sim[j, b[j]] >= cfg["sel_min"]]
        if k < len(tr):
            y = np.zeros(len(d["S"])); y[[i for _, i in d["P0"]]] = 1
            Fm.append(feats2(d["S"], X)); Lb.append(y); Ys.append(sim.max(0)); nm.append((len(d["S"]), len(d["R"])))
    F2, y, ys = np.vstack(Fm), np.concatenate(Lb), np.concatenate(Ys)
    sels = {"v1": Sel("v1", HC(max_iter=100, max_depth=6, random_state=0).fit(F2[:, :7], y)), "cls": Sel("cls", HC(max_iter=150, max_depth=6, random_state=0).fit(F2, y)),
            "reg": Sel("reg", HR(max_iter=150, max_depth=6, random_state=0).fit(F2, ys))}
    kfit = np.polyfit(np.log([n for n, _ in nm]), np.log([m for _, m in nm]), 1)
    words = lambda L, f: np.percentile([len(f(d).split()) for d in L], [50, 90]).astype(int).tolist()
    st = dict(train_docs=len(tr), dev_docs=len(dv), test_docs=len(te), doc_words_p50_p90=words(tr, lambda d: d["doc"]), summary_words_p50_p90=words(tr, lambda d: d["sum"]),
              doc_sents_p50=int(np.median([n for n, _ in nm])), summary_sents_p50=int(np.median([m for _, m in nm])), aligned_pairs=sum(len(d["P"]) for d in tr), selector_pos_rate=float(y.mean()))
    print(st, flush=True)
    mk = lambda L: [(d["S"][i], d["R"][j], f'{d["id"]}|{j}') for d in L for j, i in d["P"]]
    pr, dp = mk(tr), mk(dv); random.Random(2).shuffle(pr)
    return tr, dv, te, tv, sels, kfit, pr[:cfg["max_pairs"]], dp, st

# It takes d (one document), tv (TF-IDF vectorizer) and sel (sentence selector) as input, gets the importance score of each sentence and returns an importance score per sentence.
def scores(d, tv, sel):
    if "_f" not in d:
        X = tv.transform(d["S"]); d["_f"] = feats2(d["S"], X); G = (X @ X.T).toarray(); d["_G"] = G; d["_sc"] = {}
    if sel.kind not in d["_sc"]:
        d["_sc"][sel.kind] = sel.score(d["_f"])
    return d["_sc"][sel.kind]

# It takes d (one document), tv (TF-IDF vectorizer), sel (sentence selector), budget (word budget), lam (strength) and k (number of sentences) as input, picks important sentences, avoiding repeats and returns indices of the chosen sentences.
def rank(d, tv, sel, budget, lam, k=None):
    if not d["S"]:
        return []
    sc = scores(d, tv, sel); G = d["_G"]; wc = [len(s.split()) for s in d["S"]]; n = len(sc); out, red, used, taken = [], np.zeros(n), 0, np.zeros(n, bool)
    while len(out) < n and ((len(out) < k) if k is not None else (used < budget)):
        i = int(np.where(taken, -9e9, sc - lam * red).argmax()); taken[i] = True; out.append(i); red = np.maximum(red, G[i]); used += wc[i]
    return out

# It takes tok (tokenizer), pairs (training pairs), S (source length) and T (target length) as input, tokenizes all training pairs and returns (token arrays, loss masks, kept pairs).
def enc(tok, pairs, S, T):
    global _T
    _T = tok
    try:
        import multiprocessing as mp
        with mp.get_context("fork").Pool(4) as p:
            a = p.map(_enc1, [x[0] for x in pairs], chunksize=512); b = p.map(_enc1, [x[1] for x in pairs], chunksize=512)
    except Exception:
        a, b = [tok.encode(x[0]) for x in pairs], [tok.encode(x[1]) for x in pairs]
    ok = [r for r, v in enumerate(b) if 0 < len(v) <= T]; L = S + 1 + T + 1; X = np.full((len(ok), L), tok.pad_id, np.int64); M = np.zeros((len(ok), L), bool)
    for o, r in enumerate(ok):
        u, v = a[r][:S], b[r] + [tok.eos_id]
        X[o, S - len(u):S], X[o, S], X[o, S + 1:S + 1 + len(v)], M[o, S + 1:S + 1 + len(v)] = u, tok.eos_id, v, True
    return X, M, ok

_T = None

# It takes s (one text, read from the outer scope) as input, tokenizes one text and returns a list of token ids.
def _enc1(s):
    return _T.encode(s)

class LoRMALinear(nn.Module):
    # It takes base (base linear layer), r (rank) and alpha (scale of the low-rank update) as input, creates the small LoRMA matrices around a linear layer and returns nothing.
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.s = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    # It takes x (input tensor) as input, runs the linear layer with the LoRMA update and returns the output of the updated layer.
    def forward(self, x):
        y0 = F.linear(x, self.base.weight.to(x.dtype))
        y = y0 + self.s * F.linear(F.linear(y0, self.A.to(x.dtype)), self.B.to(x.dtype))
        return y if self.base.bias is None else y + self.base.bias.to(x.dtype)

class Adapter(nn.Module):
    # It takes blk (model block), d (model size) and r (adapter size) as input, creates a small adapter that starts as no change and returns nothing.
    def __init__(self, blk, d, r):
        super().__init__()
        self.blk, self.down, self.up = blk, nn.Linear(d, r), nn.Linear(r, d)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    # It takes x (input tensor) as input, runs the block and adds the adapter output and returns output with the adapter added.
    def forward(self, x):
        x = self.blk(x)
        return x + self.up(F.gelu(self.down(x)))

# It takes bb (pre-trained backbone), tech (fine-tuning method) and cfg (settings) as input, prepares the model for the chosen fine-tuning method and returns the trainable weights.
def adapt(bb, tech, cfg):
    lm = bb.model
    if tech in ("lorma", "adapter"):
        for p in lm.parameters():
            p.requires_grad_(False)
    if tech == "lorma":
        for n in bb.linear_names():
            par = lm; *pa, leaf = n.split(".")
            for q in pa:
                par = getattr(par, q)
            setattr(par, leaf, LoRMALinear(getattr(par, leaf), cfg["r"], cfg["alpha"]))
    if tech == "adapter":
        for i, b in enumerate(lm.blocks):
            if type(b).__name__ == "FeedForward":
                lm.blocks[i] = Adapter(b, bb.d_model, cfg["adapter_r"])
        for p in lm.norm_f.parameters():
            p.requires_grad_(True)
    return lm

class Cond(nn.Module):
    # It takes lm (language model) as input, creates the layer that adds the source text summary to the input and returns nothing.
    def __init__(self, lm):
        super().__init__()
        W = (lm.tok_emb if hasattr(lm, "tok_emb") else lm.embed.tok).weight
        self.W, self.add, d = [W], None, W.size(1)
        self.proj = nn.Linear(d, d); nn.init.zeros_(self.proj.weight); nn.init.zeros_(self.proj.bias)
        (lm.tok_emb if hasattr(lm, "tok_emb") else lm.embed).register_forward_hook(lambda m, i, o: o if self.add is None else o + self.add.to(o.dtype))

    # It takes src (source token ids) and pad (padding id) as input, sets the source text vector for the current batch and returns nothing.
    def set(self, src, pad):
        m = (src != pad).float()[..., None]
        self.add = self.proj(((self.W[0][src] * m).sum(1) / m.sum(1).clamp(min=1)).to(self.proj.weight.dtype))[:, None]

# It takes lm (language model) and x (input tensor) as input, runs the model and returns its scores.
def fwd(lm, x):
    o = lm(x)
    return o[0] if isinstance(o, tuple) else o

# It takes lm (language model) and cond (source conditioning layer) as input, copies the trainable weights and returns a dictionary of trainable weights.
def tstate(lm, cond):
    d = {k: v.detach().cpu().clone() for k, v in lm.named_parameters() if v.requires_grad}
    d.update({"cond." + k: v.detach().cpu().clone() for k, v in cond.proj.state_dict().items()})
    return d

# It takes lm (language model), cond (source conditioning layer) and d (saved weights) as input, loads saved trainable weights and returns nothing.
def load_t(lm, cond, d):
    lm.load_state_dict({k: v for k, v in d.items() if not k.startswith("cond.")}, strict=False)
    cond.proj.load_state_dict({k[5:]: v for k, v in d.items() if k.startswith("cond.")})

# It takes lm (language model), cond (source conditioning layer), tok (tokenizer), tr (training data), dp (dev pairs), cfg (settings) and name as input, trains the model to write summaries and returns (history, training time).
def train(lm, cond, tok, tr, dp, cfg, name):
    if cfg["epochs"] <= 0:
        return [], 0.0
    S, T = cfg["src"], cfg["gen"]; X, M, ok = enc(tok, tr, S, T); Xd, Md, _ = enc(tok, dp[:600], S, T); print(f"{len(ok)}/{len(tr)} training pairs kept (targets longer than {T} tokens are dropped, not cut)", flush=True)
    ps, pc = [p for p in lm.parameters() if p.requires_grad], list(cond.proj.parameters())
    opt = torch.optim.AdamW([dict(params=ps, lr=cfg["lr"]), dict(params=pc, lr=cfg["lr_cond"])], weight_decay=cfg["wd"]); base = [g["lr"] for g in opt.param_groups]
    scaler = torch.amp.GradScaler("cuda", enabled=DEV == "cuda")

    # It takes x (token ids) and m (loss mask) as input, computes the loss for one batch and returns a loss value.
    def loss_of(x, m):
        x, m = x.to(DEV), m.to(DEV)[:, 1:].reshape(-1); cond.set(x[:, :S], tok.pad_id)
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEV == "cuda"):
            lg = fwd(lm, x[:, :-1])
        return F.cross_entropy(lg.float().reshape(-1, lg.size(-1))[m], x[:, 1:].reshape(-1)[m])

    # It uses values from the outer function, computes the average validation loss and returns the average validation loss.
    @torch.no_grad()
    def nll():
        lm.eval(); v = np.mean([float(loss_of(torch.from_numpy(Xd[i:i + 64]), torch.from_numpy(Md[i:i + 64]))) for i in range(0, len(Xd), 64)]); lm.train(); return v

    sk = glob.glob(f"/kaggle/input/**/{name}_best_trainable.pt", recursive=True)
    if sk and cfg["resume"]:
        load_t(lm, cond, torch.load(sk[0], map_location="cpu")); print("resumed from", sk[0])
    rng, step, t0, hist = np.random.default_rng(cfg["seed"]), 0, time.time(), []
    total, best, bst = cfg["epochs"] * len(X) // cfg["bs"], 1e9, None
    print(f"{len(X)} training pairs, {len(ps)} trainable tensors ({sum(p.numel() for p in ps) / 1e6:.1f}M) + conditioner, {total} steps planned", flush=True)
    lm.train()
    while step < total:
        for b in np.array_split(rng.permutation(len(X)), max(1, len(X) // cfg["bs"])):
            if step == 25:
                total = min(total, int(cfg["hours"] * 3600 / ((time.time() - t0) / 25)))
            if step >= total:
                break
            f = min(1, (step + 1) / cfg["warm"]) * (0.03 + 0.97 * 0.5 * (1 + math.cos(math.pi * step / total)))
            for g, b0 in zip(opt.param_groups, base):
                g["lr"] = b0 * f
            loss = loss_of(torch.from_numpy(X[b]), torch.from_numpy(M[b]))
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(ps + pc, 1.0); scaler.step(opt); scaler.update(); step += 1
            if step % cfg["eval"] == 0 or step == total:
                v = nll(); hist.append((step, float(loss), v)); print(f"step {step}/{total} train {float(loss):.3f} dev-pair NLL {v:.4f} | {(time.time() - t0) / 60:.0f} min", flush=True)
                if v < best:
                    best, bst = v, tstate(lm, cond); torch.save(bst, f"{cfg['out']}/{name}_best_trainable.pt")
    if bst is None:
        bst = tstate(lm, cond)
    load_t(lm, cond, bst); return hist, time.time() - t0

# It takes lm (language model), cond (source conditioning layer), tok (tokenizer), srcs (source sentences), cfg (settings), lam (strength) and rep (repetition penalty) as input, generates rewritten sentences and returns generated token ids (or texts).
@torch.no_grad()
def generate(lm, cond, tok, srcs, cfg, lam=0.0, rep=1.2):
    S, T, bs, pad, eos, nbl = cfg["src"], cfg["gen"], cfg["gbs"], tok.pad_id, tok.eos_id, cfg["nbl"]; cached = hasattr(lm, "tok_emb"); lm.eval(); out = []
    for i in range(0, len(srcs), bs):
        ch = srcs[i:i + bs]; B = len(ch); x = torch.full((B, S + 1), pad, dtype=torch.long); x[:, S] = eos
        for r, s in enumerate(ch):
            a = tok.encode(s)[:S]
            if a:
                x[r, S - len(a):S] = torch.tensor(a)
        x = x.to(DEV); cond.set(x[:, :S], pad); gen, kv, cur, done, seen, mask = [], None, x, torch.zeros(B, dtype=torch.bool, device=DEV), None, None
        hist, ng = [[] for _ in range(B)], [dict() for _ in range(B)]
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEV == "cuda"):
            for t in range(T):
                if cached:
                    lg, kv = lm(cur, kv_cache=kv, pos_offset=0 if t == 0 else S + t)
                else:
                    lg = fwd(lm, torch.cat([x] + [g[:, None] for g in gen], 1))
                l = lg[:, -1].float()
                if seen is None:
                    seen = torch.zeros_like(l, dtype=torch.bool); mask = torch.zeros_like(l, dtype=torch.bool); mask.scatter_(1, x[:, :S], True); mask[:, pad] = False; mask[:, eos] = False
                if rep != 1.0:
                    l = torch.where(seen, torch.where(l > 0, l / rep, l * rep), l)
                if lam:
                    l = l + lam * mask
                l[:, pad] = -1e9
                if t < cfg["minlen"]:
                    l[:, eos] = -1e9
                if nbl:
                    for r in range(B):
                        if len(hist[r]) >= nbl - 1:
                            bn = ng[r].get(tuple(hist[r][len(hist[r]) - nbl + 1:]))
                            if bn:
                                l[r, list(bn)] = -1e9
                nx = l.argmax(-1); seen[torch.arange(B, device=DEV), nx] = True; gen.append(nx); done |= nx == eos; cur = nx[:, None]
                if nbl:
                    for r, v in enumerate(nx.tolist()):
                        hist[r].append(v)
                        if len(hist[r]) >= nbl:
                            ng[r].setdefault(tuple(hist[r][-nbl:-1]), set()).add(v)
                if done.all():
                    break
        for row in torch.stack(gen, 1).tolist():
            out.append((tok.decode(row[:row.index(eos)] if eos in row else row).strip(), eos in row))
    return out

# It takes lm (language model), cond (source conditioning layer), tok (tokenizer), docs (documents), cfg (settings), lam (strength) and rep (repetition penalty) as input, generates rewritten sentences for each document and returns nothing.
def gen_docs(lm, cond, tok, docs, cfg, lam, rep):
    flat = [d["S"][i] for d in docs for i in d["rk"]]; it = iter(generate(lm, cond, tok, flat, cfg, lam, rep)); key = (lam, rep)
    for d in docs:
        g, f = {}, {}
        for i in d["rk"]:
            g[i], f[i] = next(it)
        d.setdefault("G", {})[key], d.setdefault("F", {})[key] = g, f

# It takes h (generated texts) and r (reference texts) as input, returns the average token F1; it is the token overlap F1 between texts.
def tokf1(h, r):
    f = []
    for a, b in zip(h, r):
        a, b = Counter(a.lower().split()), Counter(b.lower().split()); o = sum((a & b).values())
        f.append(2 * o / max(1, sum(a.values()) + sum(b.values())))
    return float(np.mean(f))

# It takes d (one document), cfg (settings), key, th (threshold), strict (strict mode), mode, cand (candidate outputs) and fin (final outputs) as input, joins the chosen sentences into the final summary and returns (summary text, repeats, sentences used).
def assemble(d, cfg, key=None, th=0.0, strict=0, mode="position", cand=None, fin=None):
    out, seen, used, chosen, rep = [], set(), 0, [], 0
    G = cand if cand is not None else (d["G"][key] if key is not None else None); Fn = fin if fin is not None else (d["F"][key] if key is not None else None)
    for i in d["rk"]:
        src = d["S"][i]; g = src
        if G is not None:
            g = G[i]; w = g.split()
            if len(w) < 4 or len(set(w)) < 0.4 * len(w) or tokf1([g], [src]) < th or (strict and not Fn[i]):
                g = src
        k = re.sub(r"\W+", "", g.lower())[:80]; n = len(g.split())
        if k in seen:
            continue
        if used + n > cfg["maxw"]:
            if used >= 0.97 * cfg["maxw"]:
                break
            continue
        seen.add(k); used += n; chosen.append((i, g)); rep += g != src
    if mode == "position":
        chosen.sort(key=lambda t: t[0])
    return (" ".join(g for _, g in chosen) or d["doc"][:1500]), rep, len(chosen)

_SC = None

# It takes h (generated texts) and r (reference texts) as input, computes BERTScore, or token F1 if it is not available and returns (score, metric name).
def metric(h, r):
    global _SC
    if _SC is None:
        try:
            try:
                from bert_score import BERTScorer
            except ImportError:
                subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bert_score"]); from bert_score import BERTScorer
            _SC = BERTScorer(lang="en", batch_size=16)
        except Exception as e:
            print("!!! BERTScore unavailable (", str(e)[:100], ") - token-F1 used: turn Internet on for a real score", flush=True); _SC = False
    return (float(_SC.score(h, r)[2].mean()), "BERTScore-F1") if _SC else (tokf1(h, r), "token-F1")

# It takes path (file path) and preds (predictions) as input, writes predictions into a submission zip and returns nothing.
def write_zip(path, preds):
    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr("prediction.json", json.dumps(preds))
    with zipfile.ZipFile(path) as z:
        assert z.namelist() == ["prediction.json"]

# It takes a (text) and b (other text) as input, returns token F1; it is the token overlap F1 between two texts.
def f1(a, b):
    return tokf1([a], [b])

# It takes t (text) as input, checks whether generated text looks okay and returns True or False.
def valid(t):
    w = t.split()
    return len(w) >= 4 and len(set(w)) >= 0.4 * len(w)

# It takes src (source token ids), cs (candidates) and c (weight value) as input, picks the output that agrees most with the others and returns the chosen output.
def consensus(src, cs, c):
    ok = [x for x in cs if valid(x[0])]; pool = [(x[0], x[1], j, False) for j, x in enumerate(ok)] + [(src, True, -1, True)]; best, bs = None, -1e9
    for t, f, j, is_src in pool:
        v = sum(w * f1(t, u) for l, (u, _, w) in enumerate(ok) if l != j) + c * (1.0 if is_src else f1(t, src))
        if v > bs:
            best, bs = (t, f), v
    return best

# It takes docs (documents), dks (setting keys) and c (weight value) as input, picks the most agreed output for every sentence and returns the chosen outputs per document.
def cons_build(docs, dks, c):
    out = []
    for d in docs:
        g, f = {}, {}
        for i in d["rk"]:
            g[i], f[i] = consensus(d["S"][i], [(d["G"][k][i], d["F"][k][i], 1.0) for k in dks], c)
        out.append((g, f))
    return out

# It takes k (setting key), docs (documents), cfg (settings) and dks (setting keys) as input, gets outputs for one setting and returns outputs per document.
def part(k, docs, cfg, dks):
    if k[0][0] == "cons":
        return cons_build(docs, dks, k[0][1])
    return [(d["G"][k[0]], d["F"][k[0]]) for d in docs]

# It takes dv (dev data), te (test data), tv (TF-IDF vectorizer), sels (sentence selectors), kfit (fitted selector settings), cfg (settings) and refs (reference texts) as input, tries sentence selection settings on dev and picks the best and returns a dictionary with the best settings and all results.
def choose_selection(dv, te, tv, sels, kfit, cfg, refs):
    W = cfg["maxw"]; bud = cfg["slack"] * W; res = {}

    # It takes tag (label) and texts as input, scores one setting on dev and returns nothing.
    def ev(tag, texts):
        res[tag], mn = metric(texts, refs); print(f"dev {mn} | extractive copy only | {tag}: {res[tag]:.4f}", flush=True)

    for d in dv:
        n = len(d["S"]); K = int(np.clip(round(math.exp(kfit[1] + kfit[0] * math.log(max(n, 1)))), 3, 45))
        d["rk"] = rank(d, tv, sels["v1"], 0, 0.5, k=math.ceil(K))
    ev("previous method (fixed sentence count, document order, first 400 words)", [" ".join(" ".join(d["S"][i] for i in sorted(d["rk"])).split()[:W]) or d["doc"][:1500] for d in dv])
    orc = [sorted({i for _, i in d["P0"]}) for d in dv]
    ev("oracle ceiling (sentences really aligned to the reference)", [" ".join(" ".join(d["S"][i] for i in o).split()[:W]) or d["doc"][:1500] for d, o in zip(dv, orc)])

    # It takes kind (type), mmr (repeat penalty) and mode as input, builds summaries for one setting and returns the dev score.
    def run_cfg(kind, mmr, mode):
        for d in dv:
            d["rk"] = rank(d, tv, sels[kind], bud, mmr)
        ev(f"selector {kind} | mmr {mmr} | order {mode}", [assemble(d, cfg, mode=mode)[0] for d in dv]); return res[f"selector {kind} | mmr {mmr} | order {mode}"]

    best = (run_cfg("v1", 0.5, "position"), "v1", 0.5, "position")
    for kind in ("cls", "reg"):
        s = run_cfg(kind, 0.5, "position"); best = max(best, (s, kind, 0.5, "position"))
    for m in cfg["mmrs"]:
        if m != 0.5:
            s = run_cfg(best[1], m, best[3]); best = max(best, (s, best[1], m, best[3]))
    s = run_cfg(best[1], best[2], "rank"); best = max(best, (s, best[1], best[2], "rank"))
    print(f"chosen selection: selector {best[1]} | mmr {best[2]} | order {best[3]} | dev copy-only {best[0]:.4f}", flush=True)
    for d in dv + te:
        d["rk"] = rank(d, tv, sels[best[1]], bud, best[2])
    return dict(selector=best[1], mmr=best[2], mode=best[3], copy_only_dev=best[0], all=res)

# It takes name, A (model code module), ckpt (checkpoint path), tokf (tokenizer file path), tech (fine-tuning method) and cfg (settings) as input, runs the whole summarization pipeline and returns nothing.
def run(name, A, ckpt, tokf, tech, cfg):
    cfg = {**DEFAULT, **cfg}; out = cfg["out"]; os.makedirs(out, exist_ok=True); torch.manual_seed(cfg["seed"]); T0 = time.time()
    if cfg["fast"]:
        cfg.update(src=64, gen=48, epochs=0, align_min=0.15, teacher="", dump_teacher=0)
    tr, te = get_data()
    if cfg["smoke"]:
        tr, te = tr[:cfg["smoke"]], te[:6]; cfg.update(epochs=1 if cfg["epochs"] else 0, max_pairs=500, dev_docs=8, eval=10)
    tr, dv, te, tv, sels, kfit, pairs, dpairs, st = prepare(tr, te, cfg); refs = [d["sum"] for d in dv]
    sel = choose_selection(dv, te, tv, sels, kfit, cfg, refs)
    tp = glob.glob(f"/kaggle/input/**/{cfg['teacher']}", recursive=True) if cfg["teacher"] else []
    if tp:
        t = json.load(open(tp[0])); extra = [(s, t[k], k) for s, _, k in pairs if k in t]; pairs += extra; print(f"knowledge distillation: {len(extra)} teacher-written targets added to {len(pairs) - len(extra)} reference targets", flush=True)
    tok = A.load_tokenizer(tokf); bb = A.load_backbone(ckpt, device="cpu"); lm = adapt(bb, tech, cfg); cond = Cond(lm); lm.to(DEV); cond.to(DEV)
    assert cfg["src"] + cfg["gen"] + 2 <= getattr(bb, "max_len", 10 ** 9), f"src+gen+2 exceeds the model max_len {bb.max_len}"
    ntr = sum(p.numel() for p in lm.parameters() if p.requires_grad); nall = sum(p.numel() for p in lm.parameters())
    if cfg["epochs"] <= 0:
        sk = sorted(x for x in glob.glob(f"/kaggle/input/**/{name}_best_trainable.pt", recursive=True)) or sorted(glob.glob(f"{out}/{name}_best_trainable.pt"))
        assert sk, f"fast mode needs {name}_best_trainable.pt from a previous run in /kaggle/input"
        sd = torch.load(sk[0], map_location="cpu"); r = lm.load_state_dict({k: v for k, v in sd.items() if not k.startswith("cond.")}, strict=False); assert not r.unexpected_keys
        cond.proj.load_state_dict({k[5:]: v.float() for k, v in sd.items() if k.startswith("cond.")}); print("loaded trained weights from", sk[0], flush=True)
    hist, tsec = train(lm, cond, tok, pairs, dpairs, cfg, name)
    t1 = time.time(); res = {}
    for dk in cfg["decodes"]:
        gen_docs(lm, cond, tok, dv, cfg, *dk)
        for th in cfg["ths"]:
            for sc in cfg["stricts"]:
                r = [assemble(d, cfg, dk, th, sc, sel["mode"]) for d in dv]; res[(dk, th, sc)], mn = metric([x[0] for x in r], refs)
                print(f"dev {mn} | copy-bias {dk[0]} rep {dk[1]} | min overlap {th} | strict {sc}: {res[(dk, th, sc)]:.4f} | model-written share {sum(x[1] for x in r) / max(1, sum(x[2] for x in r)):.2f}", flush=True)
    dks = list(cfg["decodes"])
    for c in cfg["cons_c"]:
        for th in cfg["cons_ths"]:
            for sc in cfg["stricts"]:
                kk = (("cons", c), th, sc); r = [assemble(d, cfg, th=th, strict=sc, mode=sel["mode"], cand=g, fin=f) for d, (g, f) in zip(dv, cons_build(dv, dks, c))]
                res[kk], mn = metric([x[0] for x in r], refs); print(f"dev {mn} | consensus of the {len(dks)} decodings + source, weight {c} | min overlap {th} | strict {sc}: {res[kk]:.4f} | model-written share {sum(x[1] for x in r) / max(1, sum(x[2] for x in r)):.2f}", flush=True)
    top = sorted(res, key=res.get, reverse=True)[:3]; bestd = []
    for k in sorted(res, key=res.get, reverse=True):
        if k[0][0] != "cons" and k[0] not in bestd:
            bestd.append(k[0])
    need = dks if any(k[0][0] == "cons" for k in top) else list(dict.fromkeys([k[0] for k in top] + bestd[:cfg["keep_test"]]))
    print(f"best dev {mn} {res[top[0]]:.4f} | extractive copy-only with the same selection {sel['copy_only_dev']:.4f}", flush=True)
    for dk in need:
        gen_docs(lm, cond, tok, te, cfg, *dk)
    for k in top:
        dk, th, sc = k; tag = f"consensus{dk[1]}" if dk[0] == "cons" else f"bias{dk[0]}_rep{dk[1]}"
        write_zip(f"{out}/{name}_{tag}_overlap{th}_strict{sc}_dev{res[k]:.4f}_submission.zip", [{"id": d["id"], "summary": assemble(d, cfg, th=th, strict=sc, mode=sel["mode"], cand=g, fin=f)[0]} for d, (g, f) in zip(te, part(k, te, cfg, dks))])
    if cfg["dump_teacher"]:
        dk = next(k[0] for k in sorted(res, key=res.get, reverse=True) if k[0][0] != "cons"); sub = pairs[:cfg["dump_teacher"]]; g = generate(lm, cond, tok, [p[0] for p in sub], cfg, *dk)
        json.dump({p[2]: x for p, (x, fin) in zip(sub, g) if fin and len(x.split()) >= 4 and len(set(x.split())) >= 0.4 * len(x.split()) and tokf1([x], [p[0]]) >= 0.3}, open(f"{out}/p6_teacher_pairs.json", "w")); print("teacher-written targets saved:", len(g))
    dump = dict(name=name, maxw=cfg["maxw"], selection={k: v for k, v in sel.items() if k != "all"}, dev_ids=[d["id"] for d in dv], test_ids=[d["id"] for d in te], rk_dev={d["id"]: d["rk"] for d in dv}, rk_test={d["id"]: d["rk"] for d in te}, decodes={})
    for dk in cfg["decodes"]:
        bt = max(((k[1], k[2]) for k in res if k[0] == dk), key=lambda z: res[(dk,) + z]); e = dict(best_th=bt[0], best_strict=bt[1], dev_best=res[(dk,) + bt],
                                                                                                      dev={d["id"]: {str(i): [d["G"][dk][i], d["F"][dk][i]] for i in d["rk"]} for d in dv})
        if dk in need:
            e["test"] = {d["id"]: {str(i): [d["G"][dk][i], d["F"][dk][i]] for i in d["rk"]} for d in te}
        dump["decodes"][f"{dk[0]}|{dk[1]}"] = e
    json.dump(dump, open(f"{out}/{name}_gens.json", "w"))
    json.dump(dict(name=name, technique=tech, metric=mn, dev_by_setting={f"{k[0][0]}|{k[0][1]}|{k[1]}|{k[2]}": v for k, v in res.items()}, best_setting=[list(top[0][0]), top[0][1], top[0][2]], selection=sel, trainable=ntr, total=nall, params_M=nall / 1e6, train_seconds=tsec,
                   generate_seconds=time.time() - t1, history=hist, stats=st, cfg={k: v for k, v in cfg.items() if isinstance(v, (int, float, str, tuple, bool))}), open(f"{out}/{name}_summary.json", "w"), indent=1, default=str)
    zp = f"{out}/{name}_ALL_outputs.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        for f in sorted(glob.glob(f"{out}/*")):
            if os.path.isfile(f) and not f.endswith("_ALL_outputs.zip"):
                z.write(f, os.path.basename(f))
    print(zp, f"{os.path.getsize(zp) / 1e6:.1f} MB | total {(time.time() - T0) / 3600:.2f} h")

Writing p6_core.py


In [2]:
%%writefile arch_capsuleroute.py
# CapsuleRoute model code (saved as arch_capsuleroute.py)
import json, re as _re, time, math
from collections import Counter, defaultdict as _defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F

# It maps every byte to a printable character for the byte-level tokenizer and returns a dictionary from byte value to character.
def _bytes_to_unicode():

    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("\xa1"), ord("\xac") + 1)) + \
         list(range(ord("\xae"), ord("\xff") + 1))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return dict(zip(bs, [chr(c) for c in cs]))

_BYTE_ENCODER = _bytes_to_unicode()
_BYTE_DECODER = {v: k for k, v in _BYTE_ENCODER.items()}
_PRETOK_RE = _re.compile(r"\'s|\'t|\'re|\'ve|\'m|\'ll|\'d| ?[A-Za-z]+| ?[0-9]+| ?[^\sA-Za-z0-9]+|\s+")

class LegalByteBPETokenizer:

    PAD_TOKEN = "<pad>"
    EOS_TOKEN = "<|endoftext|>"

    # It creates an empty tokenizer and returns nothing.
    def __init__(self):
        self.vocab = {}
        self.id_to_token = {}
        self.merge_rank = {}
        self.pad_token_id = None
        self.eos_token_id = None
        self.padding_side = "right"
        self._cache = {}

    # It returns the vocabulary size.
    def __len__(self):
        return len(self.vocab)

    # It takes texts, target_vocab_size (target vocabulary size), sample_chars_per_doc (characters used per document) and verbose (print progress) as input, learns the BPE merges and vocabulary from the training texts and returns the trained tokenizer.
    def train(self, texts, target_vocab_size, sample_chars_per_doc=None, verbose=True):
        t0 = time.time()
        word_freq = Counter()
        for text in texts:
            if sample_chars_per_doc is not None:
                text = text[:sample_chars_per_doc]
            for m in _PRETOK_RE.finditer(text):
                sym = tuple(_BYTE_ENCODER[byte] for byte in m.group().encode("utf-8"))
                word_freq[sym] += 1

        self.vocab = {self.PAD_TOKEN: 0, self.EOS_TOKEN: 1}
        for i in range(256):
            self.vocab[_BYTE_ENCODER[i]] = len(self.vocab)
        self.pad_token_id = self.vocab[self.PAD_TOKEN]
        self.eos_token_id = self.vocab[self.EOS_TOKEN]

        words = {i: list(w) for i, w in enumerate(word_freq)}
        freqs = {i: c for i, (w, c) in enumerate(word_freq.items())}
        pair_counts = Counter()
        pair_to_words = _defaultdict(set)
        for wid, syms in words.items():
            f = freqs[wid]
            for i in range(len(syms) - 1):
                p = (syms[i], syms[i + 1])
                pair_counts[p] += f
                pair_to_words[p].add(wid)

        n_merges_needed = target_vocab_size - len(self.vocab)
        n_merges_done = 0
        while n_merges_done < n_merges_needed and pair_counts:
            max_freq = max(pair_counts.values())
            best = min(p for p, c in pair_counts.items() if c == max_freq)
            new_symbol = best[0] + best[1]
            self.merge_rank[best] = n_merges_done
            self.vocab[new_symbol] = len(self.vocab)

            for wid in list(pair_to_words.get(best, ())):
                syms, f = words[wid], freqs[wid]
                for i in range(len(syms) - 1):
                    p = (syms[i], syms[i + 1])
                    pair_counts[p] -= f
                    if pair_counts[p] <= 0:
                        pair_counts.pop(p, None)
                    pair_to_words[p].discard(wid)
                merged, i = [], 0
                while i < len(syms):
                    if i < len(syms) - 1 and syms[i] == best[0] and syms[i + 1] == best[1]:
                        merged.append(new_symbol); i += 2
                    else:
                        merged.append(syms[i]); i += 1
                words[wid] = merged
                for i in range(len(merged) - 1):
                    p = (merged[i], merged[i + 1])
                    pair_counts[p] += f
                    pair_to_words[p].add(wid)
            pair_counts.pop(best, None)
            pair_to_words.pop(best, None)
            n_merges_done += 1

        self.id_to_token = {v: k for k, v in self.vocab.items()}
        if verbose:
            print(f"    BPE training: {n_merges_done} merges learned (target {n_merges_needed}) in "
                  f"{time.time() - t0:.1f}s over {len(word_freq)} unique pretoken types "
                  f"-> final vocab size = {len(self.vocab)}")
        return self

    # It takes symbol_tuple (word as symbols) as input, applies the learned merges to one word and returns the word split into tokens (as a string).
    def _bpe(self, symbol_tuple):
        cached = self._cache.get(symbol_tuple)
        if cached is not None:
            return cached
        word = list(symbol_tuple)
        while len(word) > 1:
            best_rank, best_i = None, -1
            for i in range(len(word) - 1):
                r = self.merge_rank.get((word[i], word[i + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_i = r, i
            if best_rank is None:
                break
            word = word[:best_i] + [word[best_i] + word[best_i + 1]] + word[best_i + 2:]
        self._cache[symbol_tuple] = word
        return word

    # It takes text as input, converts text into a list of token ids and returns a list of token ids.
    def encode(self, text):
        ids = []
        for m in _PRETOK_RE.finditer(text):
            sym = tuple(_BYTE_ENCODER[byte] for byte in m.group().encode("utf-8"))
            for piece in self._bpe(sym):
                ids.append(self.vocab[piece])
        return ids

    # It takes ids (token ids) and skip_special_tokens (drop special tokens) as input, converts token ids back into text and returns text.
    def decode(self, ids, skip_special_tokens=True):
        special = {self.pad_token_id, self.eos_token_id}
        pieces = [self.id_to_token.get(int(i), "") for i in ids
                  if not (skip_special_tokens and int(i) in special)]
        raw = bytes(_BYTE_DECODER[c] for c in "".join(pieces) if c in _BYTE_DECODER)
        return raw.decode("utf-8", errors="replace")

    # It takes path (file path) as input, saves the tokenizer to a JSON file and returns nothing.
    def save(self, path):
        data = {
            "vocab": self.vocab,
            "merges": [list(pair) for pair, _rank in
                       sorted(self.merge_rank.items(), key=lambda kv: kv[1])],
            "pad_token_id": self.pad_token_id,
            "eos_token_id": self.eos_token_id,
            "padding_side": self.padding_side,
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f)

    # It takes path (file path) as input, loads a tokenizer from a JSON file and returns a tokenizer.
    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls()
        tok.vocab = data["vocab"]
        tok.id_to_token = {int(v): k for k, v in tok.vocab.items()}
        tok.merge_rank = {tuple(pair): i for i, pair in enumerate(data["merges"])}
        tok.pad_token_id = data["pad_token_id"]
        tok.eos_token_id = data["eos_token_id"]
        tok.padding_side = data.get("padding_side", "right")
        return tok

    # It takes text, max_length (max length), truncation (cut long inputs), padding (padding option) and return_tensors (return tensors) as input, tokenizes one text or a list of texts, with optional cutting and padding and returns a dictionary with input_ids and attention_mask.
    def __call__(self, text, max_length=None, truncation=False, padding=None, return_tensors=None):
        is_batch = isinstance(text, (list, tuple))
        texts = list(text) if is_batch else [text]
        all_ids = [self.encode(t) for t in texts]
        if truncation and max_length is not None:
            all_ids = [ids[:max_length] for ids in all_ids]
        lens = [len(ids) for ids in all_ids]

        if padding == "max_length" and max_length is not None:
            pad_to = max_length
        elif padding is True or padding == "longest":
            pad_to = max(lens) if lens else 0
        else:
            pad_to = None

        if pad_to is not None:
            if self.padding_side == "right":
                attn = [[1] * L + [0] * (pad_to - L) for L in lens]
                all_ids = [ids + [self.pad_token_id] * (pad_to - len(ids)) for ids in all_ids]
            else:
                attn = [[0] * (pad_to - L) + [1] * L for L in lens]
                all_ids = [[self.pad_token_id] * (pad_to - len(ids)) + ids for ids in all_ids]
        else:
            attn = [[1] * L for L in lens]

        if return_tensors == "pt":
            return {"input_ids": torch.tensor(all_ids, dtype=torch.long),
                    "attention_mask": torch.tensor(attn, dtype=torch.long)}
        return {"input_ids": all_ids if is_batch else all_ids[0],
                "attention_mask": attn if is_batch else attn[0]}

class TokenPosEmbed(nn.Module):
    # It takes vocab_size (vocabulary size), dim (model size) and max_len (max length) as input, creates the token and position embedding tables and returns nothing.
    def __init__(self, vocab_size, dim, max_len=1024):
        super().__init__()
        self.tok = nn.Embedding(vocab_size, dim)
        self.pos = nn.Embedding(max_len, dim)

    # It takes input_ids (token ids) as input, adds token and position embeddings for the input and returns embedding vectors.
    def forward(self, input_ids):
        B, T = input_ids.shape
        positions = torch.arange(T, device=input_ids.device).unsqueeze(0).expand(B, -1)
        return self.tok(input_ids) + self.pos(positions)

class FeedForward(nn.Module):
    # It takes dim (model size), mult (size multiplier) and dropout (dropout rate) as input, creates the feed-forward layer and returns nothing.
    def __init__(self, dim, mult=4, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim * mult),
            nn.GELU(),
            nn.Linear(dim * mult, dim),
            nn.Dropout(dropout),
        )
        self.norm = nn.LayerNorm(dim)

    # It takes x (input tensor) as input, runs the feed-forward layer and returns output vectors.
    def forward(self, x):
        return x + self.net(self.norm(x))

# It takes s (capsule vectors) and dim (axis) as input, squashes capsule vectors so their length stays between 0 and 1 and returns squashed vectors.
def squash(s, dim=-1):
    norm2 = (s ** 2).sum(dim=dim, keepdim=True)
    scale = norm2 / (1 + norm2)
    return scale * s / torch.sqrt(norm2 + 1e-8)

class CapsuleRoutingLayer(nn.Module):
    # It takes dim (model size), num_caps (number of capsules), window (window size) and routing_iters (routing rounds) as input, creates the capsule routing layer and returns nothing.
    def __init__(self, dim, num_caps=8, window=16, routing_iters=3):
        super().__init__()
        assert dim % num_caps == 0, "dim must be divisible by num_caps"
        self.num_caps = num_caps
        self.cap_dim = dim // num_caps
        self.window = window
        self.routing_iters = routing_iters
        self.transform = nn.Linear(dim, dim)
        self.norm = nn.LayerNorm(dim)

    # It takes x (input tensor) as input, routes information between capsules over nearby tokens and returns output vectors.
    def forward(self, x):
        B, T, D = x.shape
        W = self.window
        x_pad = F.pad(x, (0, 0, W - 1, 0))
        neigh = x_pad.unfold(1, W, 1).permute(0, 1, 3, 2)

        votes = self.transform(neigh)
        votes = votes.view(B, T, W, self.num_caps, self.cap_dim)

        b = torch.zeros(B, T, W, self.num_caps, device=x.device, dtype=x.dtype)
        v = None
        for _ in range(self.routing_iters):
            c = torch.softmax(b, dim=2)
            s = (c.unsqueeze(-1) * votes).sum(2)
            v = squash(s, dim=-1)
            agreement = (votes * v.unsqueeze(2)).sum(-1)
            b = b + agreement

        out = v.reshape(B, T, D)
        return self.norm(x + out)

class LegalCapsuleRouteLM(nn.Module):
    # It takes vocab_size (vocabulary size), dim (model size), n_layers (number of layers), num_caps (number of capsules), window (window size), routing_iters (routing rounds), max_len (max length) and dropout (dropout rate) as input, builds the full CapsuleRoute language model and returns nothing.
    def __init__(self, vocab_size, dim=512, n_layers=6, num_caps=8, window=16, routing_iters=3, max_len=1024, dropout=0.1):
        super().__init__()
        self.embed = TokenPosEmbed(vocab_size, dim, max_len)
        self.drop = nn.Dropout(dropout)
        layers = []
        for _ in range(n_layers):
            layers.append(CapsuleRoutingLayer(dim, num_caps, window, routing_iters))
            layers.append(FeedForward(dim, dropout=dropout))
        self.blocks = nn.ModuleList(layers)
        self.norm_f = nn.LayerNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size)

    # It takes input_ids (token ids) as input, runs the model and returns next-token scores.
    def forward(self, input_ids):
        x = self.drop(self.embed(input_ids))
        for blk in self.blocks:
            x = blk(x)
        return self.lm_head(self.norm_f(x))

class Tok:

    # It takes inner (inner tokenizer) as input, loads the tokenizer and returns nothing.
    def __init__(self, inner):
        self.inner = inner; self.vocab_size = len(inner)
        self.pad_id, self.eos_id, self.bos_id = inner.pad_token_id, inner.eos_token_id, None
    # It takes text as input, converts text into token ids and returns a list of token ids.
    def encode(self, text): return self.inner.encode(text)
    # It takes ids (token ids) and skip_special_tokens (drop special tokens) as input, converts token ids back into text and returns text.
    def decode(self, ids, skip_special_tokens=True): return self.inner.decode(ids, skip_special_tokens)

# It takes tok_path (tokenizer file path) as input, loads the tokenizer from its JSON file and returns a tokenizer.
def load_tokenizer(tok_path):
    return Tok(LegalByteBPETokenizer.load(tok_path))

class Backbone:
    # It takes model and cfg (settings) as input, wraps the pre-trained model with simple helper methods and returns nothing.
    def __init__(self, model, cfg):
        self.model, self.config = model, dict(cfg)
        self.d_model = int(cfg["dim"]); self.max_len = int(cfg["max_len"])
        self.n_params = sum(p.numel() for p in model.parameters())
        self.causal = True; self.bidirectional = False

    # It takes input_ids (token ids) and attention_mask (mask of real tokens) as input, returns the hidden vector of every token.
    def hidden(self, input_ids, attention_mask=None):

        m = self.model
        x = m.drop(m.embed(input_ids))
        for blk in m.blocks: x = blk(x)
        return m.norm_f(x)

    # It returns the names of the linear layers inside the blocks.
    def linear_names(self):
        return [n for n, mod in self.model.named_modules() if n.startswith("blocks.") and isinstance(mod, nn.Linear)]

    # It returns the list of model blocks.
    def block_modules(self):
        return list(self.model.blocks)

# It takes ckpt_path (checkpoint path), device, max_len (max length), num_caps (number of capsules), window (window size), routing_iters (routing rounds) and dropout (dropout rate) as input, loads the pre-trained model from its checkpoint and returns the backbone ready to use.
def load_backbone(ckpt_path, device="cpu", max_len=None, num_caps=8, window=16, routing_iters=3, dropout=0.1):

    pkg = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    sd = pkg["state_dict"] if isinstance(pkg, dict) and "state_dict" in pkg else (pkg["model"] if isinstance(pkg, dict) and "model" in pkg else pkg)
    sd = {(k[7:] if k.startswith("module.") else k): v for k, v in sd.items()}
    V, D = sd["embed.tok.weight"].shape; P = sd["embed.pos.weight"].shape[0]
    n_layers = (1 + max(int(k.split(".")[1]) for k in sd if k.startswith("blocks."))) // 2
    model = LegalCapsuleRouteLM(V, dim=D, n_layers=n_layers, num_caps=num_caps, window=window, routing_iters=routing_iters, max_len=P, dropout=dropout)
    missing, unexpected = model.load_state_dict(sd, strict=False)
    if missing or unexpected:
        raise RuntimeError(f"checkpoint does not match Legal-CapsuleRoute: missing={missing[:6]} unexpected={unexpected[:6]}")
    model.to(device)
    return Backbone(model, dict(dim=D, max_len=P, vocab_size=V, n_layers=n_layers))

Writing arch_capsuleroute.py


In [3]:
# Load student model and train it with distillation
import p6_core as C, arch_capsuleroute as A
FAST = False
CKPT = ""
TOK = ""
CKPT = C.pick(CKPT, ["*capsule*.pt", "*.pt"])
TOK = C.pick(TOK, ["*tokenizer*capsule*.json", "*tokenizer*.json", "*tok*.json"])
CFG = dict(lr=3e-4, epochs=4, hours=5.0, teacher="p6_teacher_pairs.json", fast=FAST)
C.run("p6_2_cr_kd", A, CKPT, TOK, "full", CFG)

auto-selected /kaggle/input/datasets/divyaptak/capsuleroute-final-pt/legal_capsuleroute_random_best_random_90_10 (1).pt | other matches: []
auto-selected /kaggle/input/datasets/divyaptak/tokenizer-capsuleroute/tokenizer_capsuleroute.json | other matches: []


Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

fields of the first record: ['id', 'document', 'summary']
fields of the first record: ['id', 'document']
{'train_docs': 6910, 'dev_docs': 120, 'test_docs': 100, 'doc_words_p50_p90': [3127, 7843], 'summary_words_p50_p90': [638, 1493], 'doc_sents_p50': 90, 'summary_sents_p50': 20, 'aligned_pairs': 167698, 'selector_pos_rate': 0.18522605445957857}


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

... (7 outputs left out)

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


dev BERTScore-F1 | extractive copy only | previous method (fixed sentence count, document order, first 400 words): 0.8512
dev BERTScore-F1 | extractive copy only | oracle ceiling (sentences really aligned to the reference): 0.8849
dev BERTScore-F1 | extractive copy only | selector v1 | mmr 0.5 | order position: 0.8492
dev BERTScore-F1 | extractive copy only | selector cls | mmr 0.5 | order position: 0.8497
dev BERTScore-F1 | extractive copy only | selector reg | mmr 0.5 | order position: 0.8539
dev BERTScore-F1 | extractive copy only | selector reg | mmr 0.3 | order position: 0.8552
dev BERTScore-F1 | extractive copy only | selector reg | mmr 0.7 | order position: 0.8527
dev BERTScore-F1 | extractive copy only | selector reg | mmr 0.3 | order rank: 0.8543
chosen selection: selector reg | mmr 0.3 | order position | dev copy-only 0.8552
knowledge distillation: 3 teacher-written targets added to 150000 reference targets
138457/150003 training pairs kept (targets longer than 80 tokens are 

/kaggle/working/p6_core.py:272: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  v = nll(); hist.append((step, float(loss), v)); print(f"step {step}/{total} train {float(loss):.3f} dev-pair NLL {v:.4f} | {(time.time() - t0) / 60:.0f} min", flush=True)


step 1000/17307 train 4.038 dev-pair NLL 4.0381 | 9 min
step 1500/17307 train 3.836 dev-pair NLL 3.9446 | 14 min
step 2000/17307 train 4.027 dev-pair NLL 3.8581 | 18 min
step 2500/17307 train 3.652 dev-pair NLL 3.8060 | 23 min
step 3000/17307 train 3.666 dev-pair NLL 3.7586 | 28 min
step 3500/17307 train 3.788 dev-pair NLL 3.6933 | 32 min
step 4000/17307 train 4.112 dev-pair NLL 3.6498 | 37 min
step 4500/17307 train 3.507 dev-pair NLL 3.6014 | 41 min
step 5000/17307 train 3.590 dev-pair NLL 3.5679 | 46 min
step 5500/17307 train 3.282 dev-pair NLL 3.5238 | 51 min
step 6000/17307 train 3.438 dev-pair NLL 3.5003 | 55 min
step 6500/17307 train 3.447 dev-pair NLL 3.4729 | 60 min
step 7000/17307 train 3.267 dev-pair NLL 3.4319 | 65 min
step 7500/17307 train 3.350 dev-pair NLL 3.4040 | 69 min
step 8000/17307 train 3.344 dev-pair NLL 3.3833 | 74 min
step 8500/17307 train 3.188 dev-pair NLL 3.3447 | 78 min
step 9000/17307 train 2.912 dev-pair NLL 3.3434 | 83 min
step 9500/17307 train 3.393 dev-